## Extract Junior Authors from Matched Awards
## What We Did in This Notebook

1. **Loaded Matched Awards Data**: Imported 890 matched awards from the OpenAlex dataset that were previously matched to conference best papers.

2. **Cleaned the Data**: Removed unmatched rows and reset the index for consistent tracking.

3. **Extracted Junior Authors**: Iterated through each matched award and identified junior authors (career age ≤ 5 years at time of award) among the top 3 authors of each paper.

4. **Collected Author Metadata**: For each junior author found, we gathered:
    - Author name and OpenAlex ID
    - Career age (years since first publication)
    - Total publications at the time of award
    - Affiliated institutions and countries
    - Position in authorship list
    - Whether they were corresponding author

5. **Generated Summary Statistics**: 
    - Total junior authors identified
    - Distribution across conferences
    - Career age distribution
    - Match routes used to identify papers

6. **Saved Results**: Exported the complete list of junior authors to a CSV file for further analysis.

In [ ]:
## What We Did in This Notebook

1. **Loaded Matched Awards Data**: Imported 890 matched awards from the OpenAlex dataset that were previously matched to conference best papers.

2. **Cleaned the Data**: Removed unmatched rows and reset the index for consistent tracking.

3. **Extracted Junior Authors**: Iterated through each matched award and identified junior authors (career age ≤ 5 years at time of award) among the top 3 authors of each paper.

4. **Collected Author Metadata**: For each junior author found, we gathered:
    - Author name and OpenAlex ID
    - Career age (years since first publication)
    - Total publications at the time of award
    - Affiliated institutions and countries
    - Position in authorship list
    - Whether they were corresponding author

5. **Generated Summary Statistics**: 
    - Total junior authors identified
    - Distribution across conferences
    - Career age distribution
    - Match routes used to identify papers

6. **Saved Results**: Exported the complete list of junior authors to a CSV file for further analysis.

In [1]:
import pandas as pd
import requests
import json
import time
import ast

# ── Load new matched file ──────────────────────────────────────────────────────
matched_df = pd.read_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\huang_matched_openalex.csv')

# Drop unmatched rows
matched_df = matched_df[matched_df['openalex_id'].notna()].copy()
matched_df = matched_df.reset_index(drop=True)

# Add award_id if missing
if 'award_id' not in matched_df.columns:
    matched_df['award_id'] = matched_df.index + 1

print(f"Working with {len(matched_df)} matched awards across "
      f"{matched_df['conference'].nunique()} conferences")
print(matched_df['conference'].value_counts())


Working with 890 matched awards across 30 conferences
conference
CHI           185
ICSE           88
FSE            63
UIST           34
PLDI           32
STOC           31
FOCS           30
AAAI           26
ACL            26
IJCAI          25
INFOCOM        24
SOSP           22
NSDI           21
VLDB           21
CVPR           20
KDD            20
PODS           20
OSDI           19
ICML           19
WWW            19
SIGIR          19
SIGMOD         19
CIKM           16
SIGMETRICS     15
NeurIPS        15
SODA           14
SIGCOMM        14
ICCV           12
S&P            11
MOBICOM        10
Name: count, dtype: int64


In [2]:
# ── Helper functions (unchanged from old notebook) ────────────────────────────

def get_author_works(author_id):
    """Retrieve all works for a given author with pagination."""
    all_works = []
    cursor = '*'
    while cursor:
        url = 'https://api.openalex.org/works'
        params = {
            'filter': f'author.id:{author_id}',
            'per-page': 200,
            'cursor': cursor,
            'mailto': 'shaheryar.4822@student.uu.se'  # ← replace
        }
        response = requests.get(url, params=params)
        data = response.json()
        results = data.get('results', [])
        all_works.extend(results)
        cursor = data.get('meta', {}).get('next_cursor')
        if cursor:
            time.sleep(0.05)
    return all_works

def calculate_career_age(works, reference_year):
    """Calculate years since first publication."""
    years = [w['publication_year'] for w in works if w.get('publication_year')]
    if not years:
        return None
    return reference_year - min(years)

# ── NEW: parse authorships from string format in new CSV ─────────────────────
def parse_authorships(raw):
    """Handle both JSON string and list formats."""
    if isinstance(raw, list):
        return raw
    if not isinstance(raw, str) or raw.strip() in ('', 'nan', '[]'):
        return []
    try:
        return json.loads(raw)
    except:
        try:
            return ast.literal_eval(raw)
        except:
            return []


In [3]:
# ── Main extraction loop ──────────────────────────────────────────────────────

junior_authors = []
processed_authors = set()

for idx, row in matched_df.iterrows():
    work_id     = row['openalex_id']
    award_year  = int(row['year'])
    conference  = row['conference']
    award_title = str(row.get('oa_title') or row.get('paper_title', ''))

    print(f"\n{idx+1}/{len(matched_df)} {conference} {award_year} {award_title[:50]}...")

    authorships = parse_authorships(row.get('authorships', '[]'))

    # Top 3 authors only (same rule as before)
    for position, authorship in enumerate(authorships[:3], 1):
        author     = authorship.get('author', {})
        author_id  = author.get('id')
        author_name = author.get('display_name', 'Unknown')

        if not author_id:
            print(f"  ⚠ No author ID at position {position}")
            continue

        if author_id in processed_authors:
            print(f"  {author_name} already processed")
            continue
        processed_authors.add(author_id)

        print(f"  {author_name} pos {position}...", end=' ')

        author_works = get_author_works(author_id)
        career_age   = calculate_career_age(author_works, award_year)

        if career_age is None:
            print(f"No career data")
            continue

        if career_age <= 5:
            institutions = [
                {'id': inst.get('id'), 'name': inst.get('display_name'),
                 'country': inst.get('country_code')}
                for inst in authorship.get('institutions', [])
            ]
            junior_authors.append({
                'award_id':           row['award_id'],
                'conference':         conference,
                'award_year':         award_year,
                'work_id':            work_id,
                'award_title':        award_title,
                'author_id':          author_id,
                'author_name':        author_name,
                'author_position':    position,
                'career_age_at_award': career_age,
                'total_pubs_at_award': len(author_works),
                'institutions':       json.dumps(institutions),
                'is_corresponding':   authorship.get('is_corresponding', False),
                'match_route':        row.get('match_route', ''),
            })
            print(f"✓ JUNIOR age {career_age}y, {len(author_works)} pubs")
        else:
            print(f"Senior age {career_age}y")

        time.sleep(0.1)

    # Save progress every 50 awards
    if (idx + 1) % 50 == 0:
        pd.DataFrame(junior_authors).to_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\junior_authors_progress.csv', index=False)
        print(f"\n── Progress: {len(junior_authors)} juniors found from {idx+1} awards ──\n")

# ── Final save ────────────────────────────────────────────────────────────────
junior_df = pd.DataFrame(junior_authors)
junior_df.to_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\junior_authors_all_conferences.csv', index=False)

print("=" * 60)
print(f"Junior authors found:  {len(junior_df)}")
print(f"From awards:           {len(matched_df)}")
print(f"Avg juniors/award:     {len(junior_df)/len(matched_df):.2f}")
print(f"\nAge distribution:\n{junior_df['career_age_at_award'].value_counts().sort_index()}")
print(f"\nTop conferences:\n{junior_df['conference'].value_counts().head(10)}")
print(f"\nMatch route of juniors found:\n{junior_df['match_route'].value_counts()}")



1/890 AAAI 2018 Memory-Augmented Monte Carlo Tree Search...
  Chenjun Xiao pos 1... Senior age 6y
  Jincheng Mei pos 2... ✓ JUNIOR age 4y, 29 pubs
  Martin Müller pos 3... Senior age 52y

2/890 ACL 2018 Finding syntax in human encephalography with beam ...
  John Hale pos 1... Senior age 88y
  Chris Dyer pos 2... Senior age 48y
  Adhiguna Kuncoro pos 3... ✓ JUNIOR age 2y, 35 pubs

3/890 ACL 2018 Learning to Ask Good Questions: Ranking Clarificat...
  Sudha Rao pos 1... Senior age 7y
  Hal Daumé pos 2... Senior age 17y

4/890 ACL 2018 Plurality effects in an exhaustification-based the...
  Alexandre Cremers pos 1... Senior age 7y

5/890 CHI 2018 Agile 3D Sketching with Air Scaffolding...
  Yongkwan Kim pos 1... Senior age 13y
  Sang-Gyun An pos 2... ✓ JUNIOR age 1y, 8 pubs
  Joon Hyub Lee pos 3... Senior age 6y

6/890 CHI 2018 Pinpointing...
  Mikko Kytö pos 1... Senior age 21y
  Barrett Ens pos 2... Senior age 10y
  Thammathip Piumsomboon pos 3... Senior age 7y

7/890 CHI 2018 Data I